# Comparar extracción de texto de un PDF digital

Este cuaderno extrae el texto **página por página** con PyMuPDF, pypdf y pdfplumber. El resultado es un DataFrame desplegable en Jupyter y un archivo Excel con una fila por página y una columna por librería.

Se usa por defecto el PDF digital de ejemplo incluido en esta carpeta. Puedes cambiar `PDF_PATH` por la ruta de otro PDF con texto seleccionable. Este ejemplo compara el texto que se extrae; diferencias de orden, espacios o saltos de línea no implican por una pérdida de contenido.


In [ ]:
# Si hace falta, instala las dependencias en el entorno activo:
# %pip install pymupdf pypdf pdfplumber pandas openpyxl

from pathlib import Path
import pandas as pd
import pymupdf
from pypdf import PdfReader
import pdfplumber

PDF_PATH = Path("ejemplo_cmf_digital.pdf")  # Cambia esta ruta si usar?s otro PDF
OUTPUT_XLSX = Path("comparacion_extraccion_pdf.xlsx")

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"No se encontr? el PDF: {PDF_PATH.resolve()}")
print(f"PDF: {PDF_PATH.resolve()}")


## Extraer el texto de cada página

Las tres bibliotecas usan numeración de página basada en cero en sus iteradores; aquí la mostramos desde 1 para facilitar la revisión contra el PDF. Las páginas sin texto quedan como cadena vacía.


In [ ]:
# PyMuPDF
with pymupdf.open(PDF_PATH) as doc:
    textos_pymupdf = [pagina.get_text() or "" for pagina in doc]

# pypdf
lector = PdfReader(str(PDF_PATH))
textos_pypdf = [pagina.extract_text() or "" for pagina in lector.pages]

# pdfplumber
with pdfplumber.open(PDF_PATH) as pdf:
    textos_pdfplumber = [pagina.extract_text() or "" for pagina in pdf.pages]

longitudes = {len(textos_pymupdf), len(textos_pypdf), len(textos_pdfplumber)}
if len(longitudes) != 1:
    raise ValueError("Las bibliotecas reportaron diferente número de páginas; revisar el PDF.")

resultado = pd.DataFrame({
    "Página": range(1, len(textos_pymupdf) + 1),
    "PyMuPDF": textos_pymupdf,
    "pypdf": textos_pypdf,
    "pdfplumber": textos_pdfplumber,
})
print(f"Páginas comparadas: {len(resultado)}")


## Comparación desplegable

Cada celda contiene el texto completo de esa página. En Jupyter puedes expandir el ancho de las columnas o desplazarte horizontalmente para comparar las salidas.


In [ ]:
# En Jupyter, display() presenta el DataFrame completo como una tabla desplegable.
try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_colwidth", None)
display(resultado)


## Exportar a Excel

El libro contiene una hoja `Textos por p?gina`. El ajuste de texto facilita la lectura y la primera fila y la columna de p?gina quedan inmovilizadas al desplazarse.


In [ ]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    resultado.to_excel(writer, index=False, sheet_name="Textos por pagina")
    hoja = writer.sheets["Textos por pagina"]
    hoja.freeze_panes = "B2"
    hoja.auto_filter.ref = hoja.dimensions
    hoja.column_dimensions["A"].width = 12
    for columna in ("B", "C", "D"):
        hoja.column_dimensions[columna].width = 60
    for fila in hoja.iter_rows(min_row=2, min_col=2, max_col=4):
        for celda in fila:
            celda.alignment = celda.alignment.copy(wrap_text=True, vertical="top")
    for numero_fila in range(2, hoja.max_row + 1):
        hoja.row_dimensions[numero_fila].height = 90

print(f"Excel guardado: {OUTPUT_XLSX.resolve()}")


## Revisión

Compara algunas páginas con el PDF abierto. Observa especialmente orden de lectura, encabezados, columnas, caracteres especiales y saltos de línea. Las bibliotecas extraen la capa de texto; no hacen OCR. Si una página está escaneada sin texto, los resultados pueden estar vacíos y será necesario un flujo de OCR.
